In [0]:
%sql
--creating sales summary records table
CREATE OR REPLACE TABLE retail_catalog.gold.sales_summary_daily
USING DELTA AS

SELECT
    TxnDate,
    COUNT(DISTINCT TransactionID) AS TotalTransactions,
    SUM(Quantity) AS TotalQuantity,
    ROUND(SUM(Amount), 2) AS TotalRevenue
FROM retail_catalog.silver.fact_sales
GROUP BY TxnDate;

In [0]:
%sql
--creating product_sales_summary table
CREATE OR REPLACE TABLE retail_catalog.gold.product_sales_summary
USING DELTA AS

SELECT
    p.ProductID,
    p.ProductName,
    p.Category,
    
    COUNT(f.TransactionID) AS TotalTransactions,
    SUM(f.Quantity) AS TotalUnitsSold,
    ROUND(SUM(f.Amount), 2) AS TotalRevenue,
    ROUND(AVG(f.Amount), 2) AS AvgTransactionValue

FROM retail_catalog.silver.fact_sales f

JOIN retail_catalog.silver.dim_product p
    ON f.ProductSK = p.ProductSK

GROUP BY
    p.ProductID,
    p.ProductName,
    p.Category;

In [0]:
%sql
--creating store_sales_summary table
CREATE OR REPLACE TABLE retail_catalog.gold.store_sales_summary
USING DELTA AS

SELECT
    s.StoreID,
    s.StoreName,
    s.Region,

    COUNT(f.TransactionID) AS TotalTransactions,
    SUM(f.Quantity) AS TotalQuantity,
    ROUND(SUM(f.Amount), 2) AS TotalRevenue

FROM retail_catalog.silver.fact_sales f

JOIN retail_catalog.silver.dim_store s
    ON f.StoreSK = s.StoreSK

GROUP BY
    s.StoreID,
    s.StoreName,
    s.Region;

In [0]:
%sql
--creating customer_sales_summary table
CREATE OR REPLACE TABLE retail_catalog.gold.customer_sales_summary
USING DELTA AS

SELECT
    c.CustomerID,
    c.CustomerName,
    c.City,

    COUNT(f.TransactionID) AS TotalTransactions,
    SUM(f.Quantity) AS TotalQuantity,
    ROUND(SUM(f.Amount), 2) AS LifetimeValue

FROM retail_catalog.silver.fact_sales f

JOIN retail_catalog.silver.dim_customer c
    ON f.CustomerSK = c.CustomerSK

WHERE c.IsActive = 1

GROUP BY
    c.CustomerID,
    c.CustomerName,
    c.City;

In [0]:
%sql
--creating region_sales_summary table
CREATE OR REPLACE TABLE retail_catalog.gold.region_sales_summary
USING DELTA AS

SELECT
    s.Region,

    COUNT(DISTINCT f.TransactionID) AS TotalTransactions,
    SUM(f.Quantity) AS TotalUnitsSold,
    ROUND(SUM(f.Amount), 2) AS TotalRevenue

FROM retail_catalog.silver.fact_sales f

JOIN retail_catalog.silver.dim_store s
    ON f.StoreSK = s.StoreSK

GROUP BY s.Region;

In [0]:
%sql
--total revenue
SELECT
    SUM(Amount) AS FactRevenue
FROM retail_catalog.silver.fact_sales;

In [0]:
%sql
--total revenue from sales summary
SELECT
    SUM(TotalRevenue)
FROM retail_catalog.gold.sales_summary_daily;

In [0]:
%sql
SELECT ProductID, COUNT(*)
FROM retail_catalog.gold.product_sales_summary
GROUP BY ProductID
HAVING COUNT(*) > 1;

In [0]:
%sql
SELECT *
FROM retail_catalog.gold.store_sales_summary
WHERE TotalRevenue IS NULL;